<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/04-naive-bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>

# Naive Bayes — simple examples

This notebook demonstrates two short, classroom-friendly examples of Naive Bayes classifiers in scikit-learn:

1. **GaussianNB** on the Iris dataset (numerical features).
2. **MultinomialNB** on a tiny toy text dataset (bag-of-words features).

These examples follow the concepts from the provided slides (see instructor slides for background on Bayes rule, independence assumptions, and common scikit-learn NB estimators).

In [ ]:
# Basic imports used throughout the notebook
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline


## Example 1 — GaussianNB on the Iris dataset

We use the classic Iris dataset. We'll fit a Gaussian Naive Bayes classifier, evaluate performance, and visualize a decision boundary using the first two features for clarity.

In [ ]:
# Load iris dataset
iris = datasets.load_iris()
X = iris.data[:, 2:]  # use only the first two features for easy plotting (sepal length & width)
y = iris.target
feature_names = iris.feature_names[2:]
class_names = iris.target_names

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Fit GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Predict and evaluate
y_pred = gnb.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:\n', cm)


In [ ]:
# Plot decision boundaries (using first two features only)
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = gnb.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, Z, alpha=0.3)
scatter = plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor='k', s=60)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('GaussianNB decision regions (first two features)')
plt.legend(handles=scatter.legend_elements()[0], labels=list(class_names))
plt.show()


## Example 2 — MultinomialNB on a small text dataset

MultinomialNB is commonly used for discrete/count features — e.g., bag-of-words text features. Below we create a tiny toy dataset to illustrate usage.

In [ ]:
# Tiny toy text dataset
texts = [
    'I love this course, it is great and useful',
    'This lecture is boring and dull',
    'Amazing examples and great explanations',
    'I dislike the homework, it is too hard',
    'Very helpful instructor and clear lectures',
    'Terrible slides and bad examples',
    'Clear assignments and fair grading',
    'The pace is too fast and confusing',
    'Great feedback and supportive teaching',
    'The quizzes are frustrating and unclear',
    'Engaging labs and well-structured projects',
    'The grading policy feels unfair and inconsistent',
    'Helpful office hours and prompt responses',
    'Assignments are vague and instructions are confusing',
    'Interesting topics and practical examples'
 ]
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]  # 1 = positive, 0 = negative (toy sentiment labels)

# Convert to bag-of-words counts
vec = CountVectorizer()
X_counts = vec.fit_transform(texts)

# Train-test split (keep indices to map back to text)
indices = np.arange(len(texts))
X_train_t, X_test_t, y_train_t, y_test_t, idx_train, idx_test = train_test_split(
    X_counts, labels, indices, test_size=0.33, random_state=42, stratify=labels
 )

# Fit MultinomialNB
mnb = MultinomialNB()
mnb.fit(X_train_t, y_train_t)

# Predict and evaluate
y_pred_t = mnb.predict(X_test_t)
print('Accuracy (text):', accuracy_score(y_test_t, y_pred_t))
print('\nClassification report (text):\n', classification_report(y_test_t, y_pred_t, target_names=['neg','pos']))

# Show vocabulary and example probabilities for a class
print('Vocabulary:', vec.get_feature_names_out())
proba = mnb.predict_proba(X_test_t.toarray())
print('\nPredicted probabilities for test samples:\n', proba)

# Show the corresponding text for each test sample with probabilities
print('\nTest samples with predicted probabilities:')
for idx, probs in zip(idx_test, proba):
    print(f'- "{texts[idx]}" -> P(neg)={probs[0]:.3f}, P(pos)={probs[1]:.3f}')


### Notes

- **GaussianNB** assumes numerical features are normally distributed within each class — a good baseline for low-dimensional numeric data.
- **MultinomialNB** and **BernoulliNB** are appropriate when features are counts or binary indicators (e.g., bag-of-words for text).
- Naive Bayes classifiers make the strong ("naive") assumption that features are independent given the class; despite this, they often perform well as simple baselines, especially with high-dimensional data.

Refer to the instructor slides for the derivation of the classifier, Bayes' rule, and the independence assumptions.